In [ ]:
# JERICHON import bootstrap (shared resolver in src/project_paths.py)
from pathlib import Path
import os, sys

# Locate repository from the current working directory or its parents.
_candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
for _base in list(_candidates):
    try:
        _candidates.extend(_child for _child in _base.iterdir() if _child.is_dir())
    except (OSError, PermissionError):
        pass
PROJECT_ROOT = next((p.resolve() for p in _candidates
                     if (p / "src" / "__init__.py").is_file()
                     and (p / "src" / "data" / "builders").is_dir()), None)
if PROJECT_ROOT is None:
    _env_root = os.environ.get("JERICHON_PROJECT_ROOT")
    if _env_root:
        _candidate = Path(_env_root).expanduser().resolve()
        if (_candidate / "src" / "__init__.py").is_file():
            PROJECT_ROOT = _candidate
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Не найден корень JERICHON. Откройте VS Code в папке репозитория "
        "или задайте JERICHON_PROJECT_ROOT.")
if str(PROJECT_ROOT) in sys.path:
    sys.path.remove(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))
from src.project_paths import bootstrap_project, find_project_root
PROJECT_ROOT = bootstrap_project(PROJECT_ROOT)


In [ ]:
import mplcyberpunk
import matplotlib.pyplot as plt
plt.style.use('cyberpunk')

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import logging
import matplotlib.dates as mdates
import joblib
from datetime import datetime
logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger("research")

ROOT = PROJECT_ROOT / "data"
if str(ROOT.parent) not in sys.path:
    sys.path.insert(0, str(ROOT.parent))
from src.data.builders.dollar_bars import InformationDrivenBars
from src.data.labeling.cusum import cusum_filter
from src.data.labeling.triple_barrier import triple_barrier_labeling
from src.data.builders.micro_features import MicroFeatures
from src.data.builders.build_features import FeatureBuilder

from collections import defaultdict
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
import optuna
from tqdm import tqdm

import itertools
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# В лайт-версии, которую вы сейчас видете, доступны только YDEX и MAGN
TICKER = "MAGN"

def get_available_dates(ticker: str) -> list:
    """
    Автоматически находит все доступные даты для тикера.
    Сканирует обе папки: realtime и supercandles.
    """
    dates = set()    
    # Сканируем realtime
    base_realtime = ROOT / "sample_data" / "storage" / "history" / "realtime" / ticker
    if base_realtime.exists():
        for date_dir in base_realtime.iterdir():
            if date_dir.is_dir():
                dates.add(date_dir.name)
    # Сканируем supercandles
    base_supercandles = ROOT / "sample_data" / "storage" / "history" / "supercandles" / ticker
    if base_supercandles.exists():
        for date_dir in base_supercandles.iterdir():
            if date_dir.is_dir():
                dates.add(date_dir.name)
    # Сортируем даты
    return sorted(list(dates))

def load_history(ticker: str) -> dict:
    """
    Загружает все сырые данные за все доступные даты.
    Структура: history/{realtime|supercandles}/{ticker}/{date}/{type}.parquet
    """
    result = {
        "candles": [],
        "trades": [],
        "orderbook": [],
        "tradestats": [],
        "obstats": [],
        "orderstats": [],}

    base_realtime = ROOT / "for_ml" / "storage" / "history" / "realtime"
    base_supercandles = ROOT / "for_ml" / "storage" / "history" / "supercandles"
    dates = get_available_dates(ticker)
    
    for date in dates:
        # Realtime
        rt_dir = base_realtime / ticker / date
        for key in ["candles", "trades", "orderbook"]:
            filepath = rt_dir / f"{key}.parquet"
            if filepath.exists():
                try:
                    df = pd.read_parquet(filepath)
                    df["date"] = date
                    result[key].append(df)
                except Exception as e:
                    logger.error(f"Ошибка загрузки {filepath}: {e}")

        # SuperCandles
        sc_dir = base_supercandles / ticker / date
        for key in ["tradestats", "obstats", "orderstats"]:
            filepath = sc_dir / f"{key}.parquet"
            if filepath.exists():
                try:
                    df = pd.read_parquet(filepath)
                    df["date"] = date
                    result[key].append(df)
                except Exception as e:
                    logger.error(f"Ошибка загрузки {filepath}: {e}")
    
    # Объединяем все дни
    for key in result:
        result[key] = pd.concat(result[key], ignore_index=True) if result[key] else pd.DataFrame()
    
    return result

print(f"Загрузка данных {TICKER}...\n")
data = load_history(TICKER)
# Приводим ts к datetime с московской зоной
for key, df in data.items():
    if not df.empty and 'ts' in df.columns:
        df['ts'] = pd.to_datetime(df['ts'], utc=True).dt.tz_convert('Europe/Moscow')

# Статистика
print(f"ЗАГРУЖЕНО ДАННЫХ")
for key, df in data.items():
    print(f"  {key:<12}: {len(df):>8,} записей")
print(f"\nДиапазон времени: {data['candles']['ts'].min()} — {data['candles']['ts'].max()}")
# Выводим информацию о доступных датах
available_dates = get_available_dates(TICKER)
print(f"\nДоступные даты ({len(available_dates)}):")
for date in available_dates:
    print(f"  {date}")

In [ ]:
# ============================================================
# DOLLAR BARS + CUSUM + TRIPLE BARRIER
# ============================================================

DOLLAR_THRESHOLD = 5_000_000
CUSUM_THRESHOLD = 0.001
PT_SL = (1.3, 0.4)
VERTICAL_BARRIER = 50
TARGET_WINDOW = 45

print("=" * 70)
print(f"{TICKER} — ПОСТРОЕНИЕ DOLLAR BARS")
print("=" * 70)

# ============================================================
# 1. БЕРЕМ RAW TRADES
# ============================================================

trades = data["trades"].copy()
print(f"\nRAW TRADES: {len(trades):,} записей")
print(f"Колонки: {list(trades.columns)}")

# ============================================================
# 2. ПРОВЕРЯЕМ НЕОБХОДИМЫЕ КОЛОНКИ
# ============================================================

required_cols = ["PRICE", "QUANTITY", "VALUE", "ts"]
missing = [col for col in required_cols if col not in trades.columns]
if missing:
    raise ValueError(f"В trades.parquet отсутствуют необходимые колонки: {missing}")

# ============================================================
# 3. ПРИВОДИМ ТИПЫ
# ============================================================

trades["ts"] = pd.to_datetime(trades["ts"])
trades["PRICE"] = pd.to_numeric(trades["PRICE"], errors="coerce")
trades["QUANTITY"] = pd.to_numeric(trades["QUANTITY"], errors="coerce")
trades["VALUE"] = pd.to_numeric(trades["VALUE"], errors="coerce")

# ============================================================
# 4. УДАЛЯЕМ БИТЫЕ СДЕЛКИ
# ============================================================

before = len(trades)
trades = trades.dropna(subset=["ts", "PRICE", "QUANTITY", "VALUE"])
trades = trades[(trades["PRICE"] > 0) & (trades["QUANTITY"] > 0) & (trades["VALUE"] > 0)].copy()
after = len(trades)
print(f"\nПосле очистки: {after:,} сделок")
if before != after:
    print(f"Удалено: {before - after:,}")

# ============================================================
# 5. FORENSIC CHECK
# VALUE — реальный денежный оборот сделки.
# PRICE × QUANTITY НЕ используем как денежный оборот,
# потому что QUANTITY задан в ЛОТАХ.
price_x_quantity = (trades["PRICE"] * trades["QUANTITY"]).sum()
real_value = trades["VALUE"].sum()
print("\n" + "=" * 70)
print("TURNOVER FORENSICS")
print("=" * 70)
print(f"PRICE × QUANTITY: {price_x_quantity:,.2f} RUB")
print(f"VALUE:            {real_value:,.2f} RUB")
if price_x_quantity > 0:
    ratio = real_value / price_x_quantity
    print(f"VALUE / (PRICE × QUANTITY): {ratio:.4f}")
print("\nДля dollar bars используется именно VALUE.")

# ============================================================
# 6. ПОДГОТАВЛИВАЕМ ДАННЫЕ ДЛЯ InformationDrivenBars
# InformationDrivenBars.build_dollar() ожидает:
#     price * quantity
# Поэтому превращаем VALUE обратно в эквивалентное
# количество акций: quantity_corrected = VALUE / PRICE
# Тогда: PRICE × quantity_corrected = PRICE × (VALUE / PRICE) = VALUE
trades_for_bars = pd.DataFrame({
    "ts": trades["ts"],
    "price": trades["PRICE"],
    "quantity": trades["VALUE"] / trades["PRICE"],})
trades_for_bars = trades_for_bars.dropna(subset=["ts", "price", "quantity"])
trades_for_bars = trades_for_bars[(trades_for_bars["price"] > 0) & (trades_for_bars["quantity"] > 0)].copy()

# ============================================================
# 7. ПРОВЕРКА ПЕРЕД ПОСТРОЕНИЕМ БАРОВ
# ============================================================
reconstructed_value = (trades_for_bars["price"] * trades_for_bars["quantity"]).sum()
print("\n" + "=" * 70)
print("DOLLAR VALUE CHECK")
print("=" * 70)
print(f"Исходный VALUE:       {real_value:,.2f} RUB")
print(f"Восстановленный value:{reconstructed_value:,.2f} RUB")
relative_error = abs(reconstructed_value - real_value) / real_value
print(f"Относительная ошибка:  {relative_error:.12%}")
if relative_error > 1e-10:
    raise ValueError("КРИТИЧЕСКАЯ ОШИБКА: VALUE не восстанавливается через price × quantity.")
print("✓ VALUE корректно восстанавливается")

# ============================================================
# 8. DOLLAR BARS
# ============================================================
print("\n" + "=" * 70)
print("ПОСТРОЕНИЕ DOLLAR BARS")
print("=" * 70)
print(f"Порог: {DOLLAR_THRESHOLD:,.0f} RUB")
bar_builder = InformationDrivenBars()
dollar_bars = bar_builder.build_dollar(trades_for_bars, threshold=DOLLAR_THRESHOLD)
print(f"Получено баров: {len(dollar_bars):,}")

# ============================================================
# 9. ПРОВЕРКА ОБОРОТА DOLLAR BARS
# ============================================================
print("\n" + "=" * 70)
print("BAR FORENSICS")
print("=" * 70)
if "volume" in dollar_bars.columns:
    bars_volume = dollar_bars["volume"].sum()
    print(f"Суммарный volume bars: {bars_volume:,.2f}")
if "value" in dollar_bars.columns:
    bars_value = dollar_bars["value"].sum()
    print(f"Суммарный VALUE bars: {bars_value:,.2f}")
    print(f"Исходный VALUE:       {real_value:,.2f}")

# ============================================================
# 10. ВРЕМЕННОЙ ИНДЕКС
# ============================================================
dollar_bars["ts"] = dollar_bars["close_time"]
dollar_bars["ts"] = pd.to_datetime(dollar_bars["ts"])
if dollar_bars["ts"].dt.tz is None:
    dollar_bars["ts"] = dollar_bars["ts"].dt.tz_localize("Europe/Moscow")
dollar_bars = dollar_bars.set_index("ts").sort_index()

# ============================================================
# 11. ПЕРЕИМЕНОВАНИЕ OHLC
# ============================================================
dollar_bars = dollar_bars.rename(columns={
    "open": "OPEN",
    "high": "HIGH",
    "low": "LOW",
    "close": "CLOSE",
    "volume": "VOLUME",})

# ============================================================
# 12. УНИКАЛЬНОСТЬ ИНДЕКСА
# ============================================================
dollar_bars = dollar_bars[~dollar_bars.index.duplicated(keep="first")]
dollar_bars = dollar_bars[~dollar_bars.index.isna()]
print(f"\nИндекс уникален: {dollar_bars.index.is_unique}")
print(f"Итого dollar bars: {len(dollar_bars):,}")

# ============================================================
# 13. CUSUM
# ============================================================
print("\n" + "=" * 70)
print("CUSUM")
print("=" * 70)
events_idx = cusum_filter(dollar_bars["CLOSE"], threshold=CUSUM_THRESHOLD)
print(f"Найдено событий до корректировки: {len(events_idx)}")

# ============================================================
# 14. СОГЛАСОВАНИЕ TIMEZONE
# ============================================================
if len(events_idx) > 0:
    if dollar_bars.index.tz is not None:
        if events_idx.tz is None:
            events_idx = events_idx.tz_localize(dollar_bars.index.tz)
        elif events_idx.tz != dollar_bars.index.tz:
            events_idx = events_idx.tz_convert(dollar_bars.index.tz)
    else:
        if events_idx.tz is not None:
            events_idx = events_idx.tz_localize(None)
    events_idx = events_idx[~events_idx.duplicated()]
    pos = dollar_bars.index.get_indexer(events_idx, method="nearest")
    events_idx = dollar_bars.index[pos]
    events_idx = events_idx[~events_idx.duplicated()]
    events_idx = events_idx.sort_values()
    print(f"После согласования: {len(events_idx)} событий")

# ============================================================
# 15. ПОДГОТОВКА К TRIPLE BARRIER
# ============================================================
bars_for_labeling = dollar_bars.rename(columns={
    "open": "OPEN",
    "high": "HIGH",
    "low": "LOW",
    "close": "CLOSE",})

# ============================================================
# 16. TRIPLE BARRIER
# ============================================================
print("\n" + "=" * 70)
print("TRIPLE BARRIER")
print("=" * 70)
labeled = triple_barrier_labeling(
    bars=bars_for_labeling,
    events=events_idx,
    pt_sl=PT_SL,
    vertical_barrier=VERTICAL_BARRIER,
    target_window=TARGET_WINDOW,
    target_method="ewm",)
print(f"Размечено событий: {len(labeled)}")
if not labeled.empty:
    counts = labeled["label"].value_counts()
    print("\nРаспределение меток:")
    for lbl, name, emoji in [(1, "UP (+1)", "🟢"), (-1, "DOWN (-1)", "🔴"), (0, "TIME (0)", "🟡")]:
        cnt = counts.get(lbl, 0)
        pct = 100 * cnt / len(labeled)
        print(f"{emoji} {name}: {cnt:4d} ({pct:5.1f}%)")

# ============================================================
# 17. ВИЗУАЛИЗАЦИЯ
# ============================================================
fig, axes = plt.subplots(3, 1, figsize=(18, 12))

# ------------------------------------------------------------
# График 1 — Dollar Bars + CUSUM
# ------------------------------------------------------------
ax1 = axes[0]
ax1.plot(dollar_bars.index, dollar_bars["CLOSE"], linewidth=0.8, color="C4", label="Close")
if len(events_idx) > 0:
    ax1.scatter(events_idx, dollar_bars.loc[events_idx, "CLOSE"], color="C6", s=30, zorder=5, label=f"CUSUM events ({len(events_idx)})")
ax1.set_title(f"{TICKER} — Dollar Bars + CUSUM Events", fontsize=13)
ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
ax1.xaxis.set_major_locator(mdates.DayLocator())
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha="right")

# ------------------------------------------------------------
# График 2 — Labels
# ------------------------------------------------------------

ax2 = axes[1]
if not labeled.empty:
    colors = ["C3", "C4", "C2"]
    bars_labels = ["+1 (UP)", "-1 (DOWN)", "0 (TIME)"]
    values = [counts.get(1, 0), counts.get(-1, 0), counts.get(0, 0)]
    ax2.bar(bars_labels, values, color=colors, edgecolor="black")
    for i, v in enumerate(values):
        ax2.text(i, v + max(values) * 0.02, str(v), ha="center", fontweight="bold")
ax2.set_title("Распределение меток Triple Barrier", fontsize=13)
ax2.grid(axis="y", alpha=0.3)

# ------------------------------------------------------------
# График 3 — пример события
# ------------------------------------------------------------

ax3 = axes[2]
if not labeled.empty:
    sample = labeled.sample(1).iloc[0]
    evt_time = sample.name
    t1_time = sample["t1"]
    if not isinstance(t1_time, pd.Timestamp):
        t1_time = dollar_bars.index[t1_time] if isinstance(t1_time, int) else t1_time
    mask = (dollar_bars.index >= evt_time) & (dollar_bars.index <= t1_time)
    slice_bars = dollar_bars.loc[mask]
    ax3.plot(slice_bars.index, slice_bars["CLOSE"], linewidth=1.2, color="C4", label="Close")
    ax3.axhline(y=sample["upper"], color="C3", linestyle="--", label=f"Upper ({sample['upper']:.2f})")
    ax3.axhline(y=sample["lower"], color="C4", linestyle="--", label=f"Lower ({sample['lower']:.2f})")
    ax3.axvline(x=evt_time, color="C6", linestyle=":", label="Entry")
    ax3.axvline(x=t1_time, color="C6", linestyle=":", label="Vertical barrier")
    exit_time = sample["exit_time"]
    if not isinstance(exit_time, pd.Timestamp):
        exit_time = dollar_bars.index[exit_time] if isinstance(exit_time, int) else exit_time
    ax3.scatter(exit_time, sample["exit_price"], color="C2", s=80, zorder=5, marker="X", label=f"Exit ({sample['exit_type']}, ret={sample['ret']:+.4f})")
    ax3.set_title(f"Пример события: {evt_time.strftime('%Y-%m-%d %H:%M')} → {t1_time.strftime('%Y-%m-%d %H:%M')}", fontsize=13)
    ax3.legend(loc="upper left", fontsize=8)
    ax3.grid(True, alpha=0.3)
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

plt.tight_layout()
plt.show()

In [ ]:
"""
Ячейка 3 (исправленная): Генерация признаков
"""
TRAIN_RATIO = 0.75
print("▸ Генерация признаков...")

# Приводим колонки баров к нижнему регистру (ожидается feature-модулями)
bars_lower = dollar_bars.rename(columns={"OPEN": "open", "HIGH": "high", "LOW": "low", "CLOSE": "close", "VOLUME": "volume"})

# Микроструктурные признаки
mf = MicroFeatures(spread_window=20, volatility_window=20, intensity_window=20)
bars_with_micro = mf.build_all_features(bars_lower)

# Общие признаки (техиндикаторы, лаги, скользящие)
fb = FeatureBuilder(windows=[5, 10, 20], lags=[1, 2, 3], include_micro=False)
bars_featured = fb.build_all(bars_with_micro)
print(f"  Всего колонок в расширенных барах: {bars_featured.shape[1]}")

# Удаляем нечисловые колонки (datetime, строки и т.п.)
numeric_cols = bars_featured.select_dtypes(include=[np.number]).columns
bars_featured = bars_featured[numeric_cols]
print(f"  Из них числовых: {len(numeric_cols)}")

# Прикрепляем признаки к событиям
events_df = labeled.copy()
for col in bars_featured.columns:
    events_df[f"bar_{col}"] = events_df.index.map(
        lambda t: bars_featured.loc[t, col] if t in bars_featured.index else np.nan)

events_df = events_df.dropna()
print(f"  Событий с полными признаками: {len(events_df)}")

# Бинарные метки
mask_bin = events_df["label"] != 0
feature_cols = [c for c in events_df.columns if c.startswith("bar_")]
X = events_df[feature_cols]
y = events_df["label"]

X_bin = X.loc[mask_bin]
y_bin = y.loc[mask_bin].map({1: 1, -1: 0})

print(f"  Бинарных событий: {len(X_bin)} (UP: {y_bin.sum()}, DOWN: {len(y_bin) - y_bin.sum()})")
split_idx = int(len(X_bin) * TRAIN_RATIO)
X_train, X_test = X_bin.iloc[:split_idx], X_bin.iloc[split_idx:]
y_train, y_test = y_bin.iloc[:split_idx], y_bin.iloc[split_idx:]
print(f"  Train: {len(X_train)}, Test: {len(X_test)}")

In [ ]:
"""
Ячейка 4: Сравнение моделей на train/test
"""
models = {
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=13, eval_metric="logloss", verbosity=0),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=5, random_state=13, class_weight="balanced"),
    "LightGBM": LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=13, verbose=-1),
    "CatBoost": CatBoostClassifier(iterations=200, depth=5, learning_rate=0.05, subsample=0.8, random_seed=13, verbose=0, allow_writing_files=False),}

results = []
for name, model in models.items():
    # Проверяем, что в y_train есть оба класса, иначе модель не обучится
    if len(np.unique(y_train)) < 2:
        print(f"{name:<15}: в обучающей выборке только один класс, пропускаем")
        continue
    # Обучаем на train
    model.fit(X_train, y_train)
    # Предсказываем на test
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    # Считаем метрики (только если в y_test есть оба класса)
    if len(np.unique(y_test)) < 2:
        acc = accuracy_score(y_test, y_pred)
        print(f"{name:<15}: Accuracy={acc:.4f} (ROC AUC не определён, только один класс в тесте)")
        results.append({"Model": name, "Accuracy": acc, "ROC AUC": np.nan})
        continue
    acc = accuracy_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_proba)
    results.append({"Model": name, "Accuracy": acc, "ROC AUC": roc})
    print(f"{name:<15}: Accuracy={acc:.4f}, ROC AUC={roc:.4f}")

df_res = pd.DataFrame(results).sort_values("ROC AUC", ascending=False)
print("\nСравнение моделей:")
print(df_res.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))    
x = np.arange(len(df_res))
width = 0.35
bars1 = ax.bar(x - width/2, df_res["Accuracy"], width, label="Accuracy", color="C6")
bars2 = ax.bar(x + width/2, df_res["ROC AUC"], width, label="ROC AUC", color="C3")
    
for bar in bars1:
    height = bar.get_height()
    if not np.isnan(height):
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    
for bar in bars2:
    height = bar.get_height()
    if not np.isnan(height):
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02, f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    
ax.set_ylabel("Score")
ax.set_title("Сравнение моделей на тестовой выборке")
ax.set_xticks(x)
ax.set_xticklabels(df_res["Model"])
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(axis='y', alpha=0.3)
    
plt.tight_layout()
plt.show()

In [ ]:
"""
Ячейка 5: Бэктест моделей (реалистичная торговая симуляция)
===============================================================
Реалистичный бэктест с учётом:
- Входа по сигналу модели
- Выхода по SL/TP (барьеры Triple Barrier)
- Выхода по смене сигнала (reverse) по рыночной цене
- Принудительного закрытия в 23:00
- Учёт комиссии и проскальзывания
- Реинвестирования прибыли
"""

models = {
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=13, eval_metric="logloss", verbosity=0),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=5, random_state=13, class_weight="balanced"),
    "LightGBM": LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=13, verbose=-1),
    "CatBoost": CatBoostClassifier(iterations=200, depth=5, learning_rate=0.05, subsample=0.8, random_seed=13, verbose=0, allow_writing_files=False),}

# Параметры торговли
INITIAL_CAPITAL = 100000
COMMISSION = 0.0006
SLIPPAGE = 0.0001

test_times = X_test.index
bar_close = dollar_bars["CLOSE"]
labeled_test = labeled.loc[test_times]

def run_backtest(model, X_train, y_train, X_test, y_test, labeled_test, bar_close, initial_capital=INITIAL_CAPITAL, commission=COMMISSION, slippage=SLIPPAGE):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    preds_original = np.array([-1 if p == 0 else 1 for p in preds])

    capital = initial_capital
    equity = [initial_capital]
    trades = []
    position = None  # {'side': 'long'/'short', 'entry': цена, 'entry_time': время}

    for i, (evt_time, row) in enumerate(labeled_test.iterrows()):
        signal = preds_original[i]
        if signal == 0:
            continue

        # ---- Принудительное закрытие в 23:00 ----
        if position is not None and evt_time.hour >= 23:
            close_price = bar_close.loc[evt_time]
            if position['side'] == 'long':
                pnl_pct = (close_price / position['entry'] - 1) - commission
            else:
                pnl_pct = (position['entry'] / close_price - 1) - commission
            pnl = pnl_pct * capital
            capital += pnl
            trades.append({
                'entry_time': position['entry_time'],
                'exit_time': evt_time,
                'side': position['side'],
                'entry': position['entry'],
                'exit': close_price,
                'pnl': pnl,
                'pnl_pct': pnl_pct,
                'exit_reason': 'forced_close_23'
            })
            equity.append(capital)
            position = None
            continue

        # ---- Если позиция открыта и сигнал сменился (reverse) ----
        if position is not None and signal != (1 if position['side'] == 'long' else -1):
            close_price = bar_close.loc[evt_time]
            if position['side'] == 'long':
                pnl_pct = (close_price / position['entry'] - 1) - commission
            else:
                pnl_pct = (position['entry'] / close_price - 1) - commission
            pnl = pnl_pct * capital
            capital += pnl
            trades.append({
                'entry_time': position['entry_time'],
                'exit_time': evt_time,
                'side': position['side'],
                'entry': position['entry'],
                'exit': close_price,
                'pnl': pnl,
                'pnl_pct': pnl_pct,
                'exit_reason': 'reverse_signal'
            })
            equity.append(capital)
            position = None

        # ---- Если позиция открыта и достигнут барьер SL/TP ----
        if position is not None:
            exit_type = row.get('exit_type')
            if exit_type in ('upper', 'lower', 'vertical'):
                close_price = row['exit_price']
                if position['side'] == 'long':
                    pnl_pct = (close_price / position['entry'] - 1) - commission
                else:
                    pnl_pct = (position['entry'] / close_price - 1) - commission
                pnl = pnl_pct * capital
                capital += pnl
                trades.append({
                    'entry_time': position['entry_time'],
                    'exit_time': row['exit_time'],
                    'side': position['side'],
                    'entry': position['entry'],
                    'exit': close_price,
                    'pnl': pnl,
                    'pnl_pct': pnl_pct,
                    'exit_reason': exit_type})
                equity.append(capital)
                position = None

        # ---- Открытие новой позиции ----
        if position is None and evt_time.hour < 21:
            entry_price = bar_close.loc[evt_time]
            # Учитываем проскальзывание при входе
            if signal == 1:
                entry_price *= (1 + slippage)
                side = 'long'
            else:
                entry_price *= (1 - slippage)  # для short цена входа выше
                side = 'short'
            position = {
                'side': side,
                'entry': entry_price,
                'entry_time': evt_time}

    # ---- Закрываем последнюю позицию ----
    if position is not None:
        last_row = labeled_test.iloc[-1]
        close_price = bar_close.loc[last_row.name]
        if position['side'] == 'long':
            pnl_pct = (close_price / position['entry'] - 1) - commission
        else:
            pnl_pct = (position['entry'] / close_price - 1) - commission
        pnl = pnl_pct * capital
        capital += pnl
        trades.append({
            'entry_time': position['entry_time'],
            'exit_time': last_row.name,
            'side': position['side'],
            'entry': position['entry'],
            'exit': close_price,
            'pnl': pnl,
            'pnl_pct': pnl_pct,
            'exit_reason': 'last'})
        equity.append(capital)

    # ---- Расчёт метрик ----
    if trades:
        df_trades = pd.DataFrame(trades)
        total_return = (capital / initial_capital - 1) * 100
        returns = df_trades['pnl_pct']
        sharpe = (returns.mean() / returns.std() * np.sqrt(252)) if returns.std() != 0 else 0
        equity_curve = np.array(equity)
        peak = np.maximum.accumulate(equity_curve)
        drawdown = (equity_curve - peak) / peak
        max_drawdown = drawdown.min() * 100
        win_rate = (df_trades['pnl'] > 0).mean() * 100
        profit_factor = (df_trades[df_trades['pnl'] > 0]['pnl'].sum() /
                         abs(df_trades[df_trades['pnl'] < 0]['pnl'].sum())) if (df_trades['pnl'] < 0).any() else np.inf
        df_trades['trade_num'] = range(1, len(df_trades) + 1)
    else:
        total_return = 0
        sharpe = 0
        max_drawdown = 0
        win_rate = 0
        profit_factor = 0
        df_trades = pd.DataFrame()
        equity_curve = np.array([initial_capital])

    return {
        'total_return': total_return,
        'sharpe': sharpe,
        'max_drawdown': max_drawdown,
        'win_rate': win_rate,
        'profit_factor': profit_factor,
        'trades': df_trades,
        'equity_curve': equity_curve}

# ---- Запуск бэктеста для всех моделей ----
backtest_results = {}

for name, model in models.items():
    print(f"▶ Тестирование {name}...")
    result = run_backtest(model, X_train, y_train, X_test, y_test, labeled_test, bar_close)
    backtest_results[name] = result
    df_trades = result['trades']
    print(f"  Сделок: {len(df_trades)}")
    print(f"  Доходность: {result['total_return']:.2f}%")
    print(f"  Win rate: {result['win_rate']:.1f}%")
    print(f"  Sharpe: {result['sharpe']:.2f}")
    print(f"  Max DD: {result['max_drawdown']:.2f}%")
    print(f"  Profit Factor: {result['profit_factor']:.2f}")

# ---- График Equity кривых ----
fig, ax = plt.subplots(figsize=(14, 6))

for name, res in backtest_results.items():
    if res['total_return'] != 0:
        ax.plot(res['equity_curve'], label=f"{name} (return: {res['total_return']:.2f}%)")

ax.axhline(y=INITIAL_CAPITAL, color='C4', alpha=0.5)
ax.set_title("Equity Curve — сравнение моделей (реалистичный бэктест)", fontsize=14)
ax.set_xlabel("Номер сделки")
ax.set_ylabel("Капитал (₽)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ---- Итоговая таблица ----
print("\n" + "="*70)
print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("="*70)
summary = pd.DataFrame({
    'Модель': list(backtest_results.keys()),
    'Доходность (%)': [res['total_return'] for res in backtest_results.values()],
    'Sharpe': [res['sharpe'] for res in backtest_results.values()],
    'Max DD (%)': [res['max_drawdown'] for res in backtest_results.values()],
    'Win Rate (%)': [res['win_rate'] for res in backtest_results.values()],
    'Profit Factor': [res['profit_factor'] for res in backtest_results.values()],
    'Сделок': [len(res['trades']) for res in backtest_results.values()]})
print(summary.to_string(index=False))

# Оптимизация